Step 1. 데이터 다운로드
아래 링크에서 korean-english-park.train.tar.gz 를 다운로드받아 한영 병렬 데이터를 확보합니다.
jungyeul/korean-parallel-corpora

Step 2. 데이터 정제
set 데이터형이 중복을 허용하지 않는다는 것을 활용해 중복된 데이터를 제거하도록 합니다. 데이터의 병렬 쌍이 흐트러지지 않게 주의하세요! 중복을 제거한 데이터를 cleaned_corpus 에 저장합니다.

앞서 정의한 preprocessing() 함수는 한글에서는 동작하지 않습니다. 한글에 적용할 수 있는 정규식을 추가하여 함수를 재정의하세요!

타겟 언어인 영문엔 <start> 토큰과 <end> 토큰을 추가하고 split() 함수를 이용하여 토큰화합니다. 한글 토큰화는 KoNLPy의 mecab 클래스를 사용합니다.

모든 데이터를 사용할 경우 학습에 굉장히 오랜 시간이 걸립니다. cleaned_corpus로부터 토큰의 길이가 40 이하인 데이터를 선별하여 eng_corpus와 kor_corpus를 각각 구축하세요.

Step 3. 데이터 토큰화
앞서 정의한 tokenize() 함수를 사용해 데이터를 텐서로 변환하고 각각의 tokenizer를 얻으세요! 단어의 수는 실험을 통해 적당한 값을 맞춰주도록 합니다! (최소 10,000 이상!)

❗ 주의: 난이도에 비해 데이터가 많지 않아 훈련 데이터와 검증 데이터를 따로 나누지는 않습니다.

Step 4. 모델 설계
한국어를 영어로 잘 번역해 줄 멋진 Attention 기반 Seq2seq 모델을 설계하세요! Embedding Size와 Hidden Size는 실험을 통해 적당한 값을 맞춰 주도록 합니다

Step 5. 훈련하기
훈련엔 위에서 사용한 코드를 그대로 사용하되,  eval_step() 부분이 없음에 유의합니다! 매 스텝 아래의 예문에 대한 번역을 생성하여 본인이 생각하기에 가장 멋지게 번역한 Case를 제출하세요! (Attention Map을 시각화해보는 것도 재밌을 거예요!)

# Node 2 : Project Seq2Seq로 번역기 만들기

Step 1 & 2. 데이터 다운로드 및 정제
- 한영 데이터 로드 후 중복 데이터 제거
- 정규 표현식을 사용해 노이즈 제거 및 문장 길이 제한
Step 3. 데이터 토큰화
- 형태소 분석기로 1차 분리한 한글과 영문 데이터를 SentencePiece를 활용해 토큰화 및 텐서 변환을 진행
Step 4. Seq2Seq + Attention 모델 설계
- Encoder + Decoder + Attention 매커니즘의 Seq2Seq 모델 구현
- 공통Encoder과 Seq2Seq2, Bahdanau와 Luong 매커니즘의 Attention과 Decoder 구현
Step 5. 모델 훈련 및 하이퍼 파라미터 튜닝
- Teacher Forcing 스케줄링과 하이퍼 파라미터를 조정해 모델 성능 향상

## 하이퍼 파라미터

In [1]:
VOCAB_SIZE = 10000
BATCH_SIZE = 256
EMB_DIM = 512
HID_DIM = 1024
N_LAYERS = 2
DROPOUT = 0.35
LR = 4e-4
TF_MIN = 0.5
TF_START = 1.0
TF_DECAY = 0.015

## 환경 설정 및 데이터 다운로드

In [3]:
# 라이브러리 import
import os
import re
import tarfile
from konlpy.tag import Mecab
import sentencepiece as spm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, random_split

from tqdm import tqdm
import random

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 데이터 경로 설정
dataset_dir = "./s2s_translation/datasets"
os.makedirs(dataset_dir, exist_ok=True)

tar_path = os.path.join(dataset_dir, "korean-english-park.train.tar.gz")
extract_path = os.path.join(dataset_dir, "korean-english-park.train")
os.makedirs(extract_path, exist_ok=True)

# 데이터 압축 해제
with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(path=extract_path, filter='data')

# 해제 여부 확인
for root, dirs, files in os.walk(extract_path):
    for file in files:
        print(file)

korean-english-park.train.en
korean-english-park.train.ko


## 데이터 정제

In [5]:
# 데이터 로드
kor_file = os.path.join(extract_path, "korean-english-park.train.ko")
eng_file = os.path.join(extract_path, "korean-english-park.train.en")

with open(kor_file, "r", encoding="utf-8") as f:
    kor_lines = f.read().splitlines()

with open(eng_file, "r", encoding="utf-8") as f:
    eng_lines = f.read().splitlines()

print("kor 문장 수:", len(kor_lines))
print("eng 문장 수:", len(eng_lines))

kor 문장 수: 94123
eng 문장 수: 94123


In [6]:
# 데이터 정제 함수
def preprocess_sentence(sentence, is_kor=True):
    # 영어라면 소문자로
    if not is_kor:
        sentence = sentence.lower()

    sentence = sentence.strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)

    if is_kor:
        # 한글 정규식
        sentence = re.sub(r"[^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z0-9\u4e00-\u9fff?.!,%()$ ]", "", sentence)
    else:
        # 영어 정규식
        sentence = re.sub(r"[^a-zA-Z0-9?.!,'%()$ -]", "", sentence)
    
    # 다중 공백 하나로
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence.strip()

In [7]:
pairs = []
for kor, eng in zip(kor_lines, eng_lines):
    # 데이터 정규식 적용
    kor = preprocess_sentence(kor, is_kor=True)
    eng = preprocess_sentence(eng, is_kor=False)

    # 노이즈 제거
    if kor and eng:
        pairs.append((kor, eng))

# 중복 제거
cleaned_corpus = list(set(pairs))

mecab = Mecab()
kor_corpus = []
eng_corpus = []

for kor, eng in cleaned_corpus:
    # 형태소 분석
    kor_tokens = mecab.morphs(kor)
    eng_tokens = eng.split()

    # 문장 길이 제한
    if len(kor_tokens) <= 40 and len(eng_tokens) <= 40:
        kor_corpus.append(" ".join(kor_tokens))
        eng_corpus.append(" ".join(eng_tokens))

print("정제 후 kor 문장 수:", len(kor_corpus))
print("정제 후 eng 문장 수:", len(eng_corpus))

정제 후 kor 문장 수: 62134
정제 후 eng 문장 수: 62134


## SentencePiece 단어장 생성

In [8]:
# txt파일 경로 지정
kor_corpus_file = os.path.join(dataset_dir, "kor_corpus.txt")
eng_corpus_file = os.path.join(dataset_dir, "eng_corpus.txt")
spm_path = os.path.join(dataset_dir, "spm")
os.makedirs(spm_path, exist_ok=True)

# spm경로 지정
encoder_spm = os.path.join(spm_path, "encoder_spm")
decoder_spm = os.path.join(spm_path, "decoder_spm")

#  txt파일 쓰기
with open(kor_corpus_file, "w", encoding="utf-8") as f:
    for sentence in kor_corpus:
        f.write(sentence + "\n")

with open(eng_corpus_file, "w", encoding="utf-8") as f:
    for sentence in eng_corpus:
        f.write(sentence + "\n")

In [ ]:
# 토크나이저 학습 함수
def train_spm(input_file, model_prefix, vocab_size):
    spm.SentencePieceTrainer.Train(
        input=input_file,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        pad_id=0,
        bos_id=1,
        eos_id=2,
        unk_id=3,
        model_type="unigram"
    )

In [ ]:
# 토크나이저 학습
train_spm(kor_corpus_file, encoder_spm, VOCAB_SIZE)
train_spm(eng_corpus_file, decoder_spm, VOCAB_SIZE)

# 토크나이저 객체 생성 및 로드
encoder_tokenizer = spm.SentencePieceProcessor()
decoder_tokenizer = spm.SentencePieceProcessor()

encoder_tokenizer.Load(encoder_spm + ".model")
decoder_tokenizer.Load(decoder_spm + ".model")

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./s2s_translation/datasets/kor_corpus.txt
  input_format: 
  model_prefix: ./s2s_translation/datasets/spm/encoder_spm
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <p

True

s=120238 num_tokens/piece=8.14069
unigram_model_trainer.cc(618) LOG(INFO) EM sub_iter=1 size=14770 obj=8.25183 num_tokens=120238 num_tokens/piece=8.14069
unigram_model_trainer.cc(618) LOG(INFO) EM sub_iter=0 size=11077 obj=8.35381 num_tokens=131525 num_tokens/piece=11.8737
unigram_model_trainer.cc(618) LOG(INFO) EM sub_iter=1 size=11077 obj=8.32849 num_tokens=131510 num_tokens/piece=11.8723
unigram_model_trainer.cc(618) LOG(INFO) EM sub_iter=0 size=11000 obj=8.33047 num_tokens=131861 num_tokens/piece=11.9874
unigram_model_trainer.cc(618) LOG(INFO) EM sub_iter=1 size=11000 obj=8.32964 num_tokens=131865 num_tokens/piece=11.9877
trainer_interface.cc(689) LOG(INFO) Saving model: ./s2s_translation/datasets/spm/decoder_spm.model
trainer_interface.cc(701) LOG(INFO) Saving vocabs: ./s2s_translation/datasets/spm/decoder_spm.vocab


## 토큰화

In [ ]:
max_len = 40

# 토큰화해서 bos, eos추가 후 텐서 변환 함수
def encode_sentence(tokenizer, sentence, max_len=40):
    token_ids = tokenizer.EncodeAsIds(sentence)
    token_ids = [tokenizer.bos_id()] + token_ids + [tokenizer.eos_id()]

    # 최대 길이를 넘으면 자르고 마지막 토큰을 eos로
    if len(token_ids) > max_len:
        token_ids = token_ids[:max_len]
        token_ids[-1] = tokenizer.eos_id()

    return torch.tensor(token_ids, dtype=torch.long)

In [12]:
# 문장을 입력용 텐서로 변환
enc_corpus = [encode_sentence(encoder_tokenizer, sent, max_len) for sent in kor_corpus]
dec_corpus = [encode_sentence(decoder_tokenizer, sent, max_len) for sent in eng_corpus]

# 패딩 처리
enc_tensor = pad_sequence(
    enc_corpus,
    batch_first=True,
    padding_value=encoder_tokenizer.pad_id()
)

dec_tensor = pad_sequence(
    dec_corpus,
    batch_first=True,
    padding_value=decoder_tokenizer.pad_id()
)

## 데이터셋 구성

In [ ]:
# Dataset 상속해 Custom
class TranslationDataset(Dataset):
    def __init__(self, src_tensor, tgt_tensor):
        self.src = src_tensor
        self.tgt = tgt_tensor

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]

In [ ]:
# Dataset 객체 생성
dataset = TranslationDataset(enc_tensor, dec_tensor)

dataset_size = len(dataset)
val_size = int(dataset_size * 0.1)
train_size = dataset_size - val_size

# train, validation dataaset 분할
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# DataLoader 객체 생성
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

In [15]:
print(f"전체 데이터 개수: {dataset_size}개")
print(f"훈련 데이터 로더 크기: {len(train_loader)} 배치")
print(f"검증 데이터 로더 크기: {len(val_loader)} 배치")

전체 데이터 개수: 62134개
훈련 데이터 로더 크기: 218 배치
검증 데이터 로더 크기: 24 배치


## 모델 구성

In [ ]:
# Encoder(공통) 설계
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers=1, dropout=0.3):
        super().__init__()
        # embadding layer
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        # 다층 gru 기반 rnn
        self.gru = nn.GRU(
            emb_dim,
            hid_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        # dropout layer
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # embedding vector 변환 후 dropout 적용
        embedded = self.dropout(self.embedding(src))
        # gru layer에 embbeing seq. 통과
        outputs, hidden = self.gru(embedded)
        # 시점별 출력과 최종 hidden state 반환
        return outputs, hidden

### Bahdanau Attention

In [ ]:
# Bahdanau Attention 설계
class BahdanauAttention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        # 가중치 학습 layer
        self.W1 = nn.Linear(hid_dim, hid_dim)                           # encoder 출력값 변환
        self.W2 = nn.Linear(hid_dim, hid_dim)                           # decoder 이전 hidden state 변환
        self.v = nn.Linear(hid_dim, 1, bias=False)                      # 두값을 더한 뒤 압축할 layer

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[1]                              # encoder에서 넘어온 seq. length 
        hidden = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)          # decoder의 최상단 hidden state를 가져와서 encoder의 len.만큼 복사 [Batch, seq. len., Hidden]
        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden)) # encoder와 decoder를 더해 연관성 수치화
        attention = self.v(energy).squeeze(2)                           # hidden dim을 더해 압축하고 squeeze를 통해 제거 [Batch, seq. len.]
        return torch.softmax(attention, dim=1)                          # softmax 적용 후 반환

# Bahdanau Decoder 설계
class BahdanauDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, attention, n_layers=1, dropout=0.3):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention

        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.gru = nn.GRU(
            emb_dim,
            hid_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.fc_out = nn.Linear(hid_dim * 2, output_dim)                # attenton을 통해 얻은 context vector와 decoder gru의 hidden dim을 concat하여 사용
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, encoder_outputs):
        input_token = input_token.unsqueeze(1)                          # [Batch, 1]
        embedded = self.dropout(self.embedding(input_token))            # [Batch, 1, emb. dim.]

        attn_weights = self.attention(hidden, encoder_outputs).unsqueeze(1) # [Batch, 1, seq. len.]
        context = torch.bmm(attn_weights, encoder_outputs)              # 가중치가 곱해진 context vector 계산 [Batch, 1, seq. len.] * [Batch, seq. len., hid. dim.] = [Batch, 1, hid. dim.]

        output, hidden = self.gru(embedded, hidden)                     # emb.과 hidden state을 통해 현재 hidden state 계산 [Batch, 1, hid. dim.]

        # 1인 차원 제거
        output = output.squeeze(1)
        context = context.squeeze(1)

        prediction = self.fc_out(torch.cat((output, context), dim=1))   # concat후 fc를 통해 확률 계산 [Batch, out. dim.(Vocab Size)]
        return prediction, hidden, attn_weights                         # 예측값, 현재 hiddin state, 가중치 반환

### Luong Attention

In [ ]:
# Luong Attention 설계
class LuongAttention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.W = nn.Linear(hid_dim, hid_dim, bias=False)    # 가중치 학습 layer(encoder 출력값 변환)

    def forward(self, hidden, encoder_outputs):
        hidden = hidden[-1].unsqueeze(2)                    # decoder 상단 2개 차원 [Batch, hid. dim., 1]
        Wh = self.W(encoder_outputs)                        # 가중치 W곱하기 [Batch, seq. len., hid. dim.]
        attention = torch.bmm(Wh, hidden).squeeze(2)        # dot product 연산 후 [Batch, seq. len., 1]에서 1인 차원 제거
        return torch.softmax(attention, dim=1)              # softmax 적용 후 반환

# Luong Decoder 설계
class LuongDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, attention, n_layers=1, dropout=0.3):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention

        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.gru = nn.GRU(
            emb_dim,
            hid_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.wc = nn.Linear(hid_dim * 2, hid_dim)           # context vector와 hidden state concat후 hid. dim.으로 압축
        self.fc_out = nn.Linear(hid_dim, output_dim)        # 최종 확률 계산 layer
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, encoder_outputs):
        input_token = input_token.unsqueeze(1)              # [Batch, 1]
        embedded = self.dropout(self.embedding(input_token)) # [Batch, 1, emb. dim.]

        output, hidden = self.gru(embedded, hidden)         # 현재 hidden state 계산 [Batch, 1, hid. dim.]

        attn_weights = self.attention(hidden, encoder_outputs).unsqueeze(1) # 현재 시점의 hidden state로 attention 가중치 계산 [Batch, 1, seq. len.]
        context = torch.bmm(attn_weights, encoder_outputs)  # 가중치가 곱해진 context vector 계산

        output = output.squeeze(1)                          # [Batch, hid. dim.]
        context = context.squeeze(1)                        # [Batch, hid. dim.]

        prediction = self.fc_out(torch.tanh(self.wc(torch.cat((output, context), dim=1)))) # [Batch, out. dim.(Vocab Size)]
        return prediction, hidden, attn_weights             # 예측값, 현재 hidden state, 가중치 반환

In [ ]:
# Seq2Seq 모델 설계
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    # src: 입력 문장 텐서 [Batch, seq. len.]
    # trg: 정답 문장 텐서 (test시 None)
    def forward(self, src, trg=None, teacher_forcing_ratio=0.5, max_len=40, bos_id=1, eos_id=2):
        batch_size = src.shape[0]
        trg_vocab_size = self.decoder.output_dim
        attentions = []

        # encoder 통과
        encoder_outputs, hidden = self.encoder(src)

        # train 모드
        if trg is not None:
            trg_len = trg.shape[1]
            # 예측을 저장한 빈 텐서
            outputs = torch.zeros(batch_size, trg_len, trg_vocab_size, device=self.device)

            # 디코더의 첫입력으로 <bos>를 넣어준다.
            input_token = trg[:, 0]

            for t in range(1, trg_len):
                # 디코더 연산 수행 
                output, hidden, attn_weights = self.decoder(input_token, hidden, encoder_outputs)
                outputs[:, t] = output
                attentions.append(attn_weights)
                
                # 예측된 단어 중 가장 확률이 높은 단어 인덱스 추출
                top1 = output.argmax(1)
                
                # Teacher forcing
                # 확률에 따라 정답이나 예측값 전달 결정
                teacher_force = random.random() < teacher_forcing_ratio
                input_token = trg[:, t] if teacher_force else top1
        # test 모드
        else:
            outputs = []
            # <bos>로 채워진 텐서를 만들어 첫 입력으로 사용
            input_token = torch.full((batch_size,), bos_id, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)

            # 길이 생성 제한
            for _ in range(max_len):
                # 디코더 연산 수행
                output, hidden, attn_weights = self.decoder(input_token, hidden, encoder_outputs)
                outputs.append(output.unsqueeze(1))
                attentions.append(attn_weights)

                # 예측값을 다음 입력으로
                top1 = output.argmax(1)
                input_token = top1

                # <eos>토큰 확인 후 for문 종료
                finished |= (top1 == eos_id)
                if finished.all():
                    break
            # 예측값들을 하나로 이어 붙인다. [Batch, Time, Vocab size]
            outputs = torch.cat(outputs, dim=1)
            
        # attention가중치를 이어 붙여 반환
        attentions = torch.cat(attentions, dim=1)

        return outputs, attentions

In [20]:
# Attention 선택
def build_attention(attention_type, hid_dim):
    attention_type = attention_type.lower()

    if attention_type == "bahdanau":
        return BahdanauAttention(hid_dim)
    elif attention_type == "luong":
        return LuongAttention(hid_dim)

In [21]:
# Decoder 선택
def build_decoder(attention_type, output_dim, emb_dim, hid_dim, attention, n_layers=1, dropout=0.3):
    attention_type = attention_type.lower()

    if attention_type == "bahdanau":
        return BahdanauDecoder(
            output_dim=output_dim,
            emb_dim=emb_dim,
            hid_dim=hid_dim,
            attention=attention,
            n_layers=n_layers,
            dropout=dropout
        )
    elif attention_type == "luong":
        return LuongDecoder(
            output_dim=output_dim,
            emb_dim=emb_dim,
            hid_dim=hid_dim,
            attention=attention,
            n_layers=n_layers,
            dropout=dropout
        )

In [22]:
# 모델 생성 및 반환
def build_model(
    attention_type,
    input_dim,
    output_dim,
    emb_dim,
    hid_dim,
    n_layers,
    dropout,
    device
):
    encoder = Encoder(
        input_dim=input_dim,
        emb_dim=emb_dim,
        hid_dim=hid_dim,
        n_layers=n_layers,
        dropout=dropout
    )

    attention = build_attention(attention_type, hid_dim)

    decoder = build_decoder(
        attention_type=attention_type,
        output_dim=output_dim,
        emb_dim=emb_dim,
        hid_dim=hid_dim,
        attention=attention,
        n_layers=n_layers,
        dropout=dropout
    )

    model = Seq2Seq(encoder, decoder, device).to(device)
    return model

In [23]:
INPUT_DIM = encoder_tokenizer.GetPieceSize()
OUTPUT_DIM = decoder_tokenizer.GetPieceSize()

# 모델 생성
model = build_model(
    attention_type="luong", # bahdanau or luong
    input_dim=INPUT_DIM,
    output_dim=OUTPUT_DIM,
    emb_dim=EMB_DIM,
    hid_dim=HID_DIM,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
    device=device
)

print(model)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(10000, 512, padding_idx=0)
    (gru): GRU(512, 1024, num_layers=2, batch_first=True, dropout=0.35)
    (dropout): Dropout(p=0.35, inplace=False)
  )
  (decoder): LuongDecoder(
    (attention): LuongAttention(
      (W): Linear(in_features=1024, out_features=1024, bias=False)
    )
    (embedding): Embedding(10000, 512, padding_idx=0)
    (gru): GRU(512, 1024, num_layers=2, batch_first=True, dropout=0.35)
    (wc): Linear(in_features=2048, out_features=1024, bias=True)
    (fc_out): Linear(in_features=1024, out_features=10000, bias=True)
    (dropout): Dropout(p=0.35, inplace=False)
  )
)


## 학습 준비

In [ ]:
# 손실 함수와 최적화 알고리즘 정의
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=LR)

In [ ]:
# 단일 epoch 학습 함수
def train_epoch(model, dataloader, optimizer, criterion, epoch, teacher_forcing_ratio=0.5):
    # train 모드
    model.train()
    epoch_loss = 0

    # 진행상황 시각화
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=True)

    # mini batch 단위로 데이터 순회
    for src, trg in progress_bar:
        # Gpu 메모리로 이동
        src = src.to(device)
        trg = trg.to(device)

        # 기울기 초기화
        optimizer.zero_grad()

        # 순전파
        output, _ = model(src, trg, teacher_forcing_ratio=teacher_forcing_ratio)

        # [Batch*Time, Vocab]
        output = output[:, 1:].reshape(-1, output.shape[-1])    # bos제외
        trg = trg[:, 1:].reshape(-1)

        # 손실 계산 및 역전파
        loss = criterion(output, trg)
        loss.backward()

        # 기울기 변화량 제한
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        
        # 가중치 업데이트
        optimizer.step()

        # 현재 Batch의 손실값 누적 후 출력
        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    # 평균 손실값 반환
    return epoch_loss / len(dataloader)

In [26]:
# 예시 문장
test_sentences = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]

In [ ]:
# 추론(번역) 함수
def translate_sentence(sentence, encoder_tokenizer, decoder_tokenizer, model, max_len=40):
    # test 모드
    model.eval()

    # 데이터 전처리 및 토큰화
    sentence = preprocess_sentence(sentence, is_kor=True)
    tokens = mecab.morphs(sentence)
    sentence = " ".join(tokens)

    # 정수 id변환
    src_ids = encoder_tokenizer.EncodeAsIds(sentence)
    src_ids = [encoder_tokenizer.bos_id()] + src_ids + [encoder_tokenizer.eos_id()]
    src_tensor = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)    # [Batch, 1]

    # 모델 추론(역전파 방지)
    with torch.no_grad():
        outputs, attentions = model(
            src_tensor,
            trg=None,   # 정답X
            max_len=max_len,
            bos_id=decoder_tokenizer.bos_id(),
            eos_id=decoder_tokenizer.eos_id()
        )

    # 확률이 가장 높은 단어 추출
    pred_ids = outputs.argmax(2).squeeze(0).tolist()

    # 토큰 후처리 및 복원
    decoded_ids = []
    for idx in pred_ids:
        # <eos> 만날시 for문 종료
        if idx == decoder_tokenizer.eos_id():
            break
        # <pad>나 <unk> 필터링
        if idx not in [decoder_tokenizer.pad_id(), decoder_tokenizer.unk_id()]:
            decoded_ids.append(idx)
    # 텍스트로 변환하여 반환
    decoded_sentence = decoder_tokenizer.DecodeIds(decoded_ids)
    return decoded_sentence, attentions

In [ ]:
# 검증 함수
def evaluate(model, dataloader, criterion):
    # test 모드
    model.eval()
    epoch_loss = 0

    # 가중치 계산 비활성화
    with torch.no_grad():
        for src, trg in dataloader:
            src = src.to(device)
            trg = trg.to(device)
            # 순전파
            output, _ = model(src, trg, teacher_forcing_ratio=0.0)
            # 텐서 평탄화
            output = output[:, 1:].reshape(-1, output.shape[-1])
            trg = trg[:, 1:].reshape(-1)
            # 오차 계싼
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    # 평균 손실값 반환
    return epoch_loss / len(dataloader)

In [ ]:
# 과적합 테스트
# 배치 1개만 로드
single_src, single_trg = next(iter(train_loader))

# 배치에서 2문장만 떼어내서 테스트
micro_src = single_src[:2].to(device)
micro_trg = single_trg[:2].to(device)

model.train()
print("=== 미니 배치 2문장 과적합 테스트 시작 ===")
for i in range(200):
    optimizer.zero_grad()
    
    # 확실한 암기를 위해 정답을 100% 알려줍니다 (ratio=1.0)
    output, _ = model(micro_src, micro_trg, teacher_forcing_ratio=1.0)
    
    # Loss 계산
    output = output[:, 1:].reshape(-1, output.shape[-1])
    trg_flat = micro_trg[:, 1:].reshape(-1)
    
    loss = criterion(output, trg_flat)
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
    
    optimizer.step()
    
    if (i + 1) % 40 == 0:
        print(f"Iteration {i+1} | Loss: {loss.item():.4f}")

=== 미니 배치 2문장 과적합 테스트 시작 ===
Iteration 40 | Loss: 0.7490
Iteration 80 | Loss: 0.0376
Iteration 120 | Loss: 0.0092
Iteration 160 | Loss: 0.0050
Iteration 200 | Loss: 0.0034


## 모델 학습 및 추론

In [ ]:
# 하이퍼 파라미터 재확인
print(f"VOCAB_SIZE = {VOCAB_SIZE}")
print(f"BATCH_SIZE = {BATCH_SIZE}")
print(f"EMB_DIM = {EMB_DIM}")
print(f"HID_DIM = {HID_DIM}")
print(f"N_LAYERS = {N_LAYERS}")
print(f"DROPOUT = {DROPOUT}")
print(f"LR = {LR}")
print(f"TF_MIN = {TF_MIN}")
print(f"TF_START = {TF_START}")
print(f"TF_DECAY = {TF_DECAY}")

VOCAB_SIZE = 10000
BATCH_SIZE = 256
EMB_DIM = 512
HID_DIM = 1024
N_LAYERS = 2
DROPOUT = 0.35
LR = 0.0004
TF_MIN = 0.5
TF_START = 1.0
TF_DECAY = 0.015


In [ ]:
EPOCHS = 40
CHECK = 5
SAVE = 2

saved_results = []

# 모델 가중치 저장 폴더
model_dir = os.path.join(dataset_dir, "model")
os.makedirs(model_dir, exist_ok=True)

# Epoch만큼 학습
for epoch in range(EPOCHS):
    # Teacher Forcing ratio scheduling
    tf_ratio = max(TF_MIN, TF_START - epoch * TF_DECAY)
    
    # train and test
    train_loss = train_epoch(model, train_loader, optimizer, criterion, epoch, teacher_forcing_ratio=tf_ratio)
    val_loss = evaluate(model, val_loader, criterion)
    
    # Epoch당 결과 출력
    print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

    # Check epoch마다 테스트 문장 번역 후 결과 기록
    if (epoch + 1) % CHECK == 0:
        epoch_result = {
            "epoch": epoch + 1,
            "loss": train_loss,
            "translations": []
        }
        print(f'Epoch {epoch+1}')

        for i, sentence in enumerate(test_sentences, 1):
            # test 모드로 변역
            translation, attentions = translate_sentence(
                sentence,
                encoder_tokenizer,
                decoder_tokenizer,
                model,
                max_len=40
            )
            print(f'{translation}')

            # 결과 dict로 저장
            epoch_result["translations"].append({
                "source": sentence,
                "prediction": translation,
                "attentions": attentions
            })

        saved_results.append(epoch_result)

    # Save마다 모델의 가중치 저장
    if (epoch + 1) % SAVE == 0:
        save_path = os.path.join(model_dir, f'seq2seq_model_epoch_{epoch+1:02d}.pt')
        torch.save(model.state_dict(), save_path)

Epoch 1: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.84]


Epoch 1/40 | Train Loss: 5.0111 | Val Loss: 8.8608


Epoch 2: 100%|██████████| 218/218 [02:24<00:00,  1.51it/s, loss=4.82]


Epoch 2/40 | Train Loss: 4.7972 | Val Loss: 8.5538


Epoch 3: 100%|██████████| 218/218 [02:24<00:00,  1.51it/s, loss=4.66]


...

Epoch 38/40 | Train Loss: 3.7606 | Val Loss: 6.6805


Epoch 39: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=3.38]


Epoch 39/40 | Train Loss: 3.7054 | Val Loss: 6.7545


Epoch 40: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=3.54]


Epoch 40/40 | Train Loss: 3.6791 | Val Loss: 6.7740
Epoch 40
obama iss obama's president , the , obama said he is the best president to help obama .
thessss the streets , the-yardss .
no one has beeneded for coffee ,s , and thess .
the least seven people died when the tornado hit the killed in the tornado , which was reported in the .


In [ ]:
# 저장된 test(번역) 문장 확인
for result in saved_results:
    print(f"\nEpoch {result['epoch']:02d} | Loss: {result['loss']:.4f}")
    for i, item in enumerate(result["translations"], 1):
        print(f"K{i}) {item['source']}")
        print(f"E{i}) {item['prediction']} <end>")


Epoch 05 | Loss: 4.3920
K1) 오바마는 대통령이다.
E1) obama's campaign is expected to be the president's president , obama said . <end>
K2) 시민들은 도시 속에 산다.
E2) they're not going to get out of thes , and they're going to get out of the . <end>
K3) 커피는 필요 없다.
E3) it's not a very important , and i'm going to be a lot of people . <end>
K4) 일곱 명의 사망자가 발생했다.
E4) the death toll was found in the , but the number of people died in the city . <end>

Epoch 10 | Loss: 4.0468
K1) 오바마는 대통령이다.
E1) obama's president is going to be president barack obama's president , obama said . <end>
K2) 시민들은 도시 속에 산다.
E2) the victims are in the streets , and they're seen as a result of theing thes . <end>
K3) 커피는 필요 없다.
E3) it's not a very difficult , and i'm not going to get a , said a statement . <end>
K4) 일곱 명의 사망자가 발생했다.
E4) the death toll was killed and one person died in the deaths of the deaths . <end>

Epoch 15 | Loss: 3.8703
K1) 오바마는 대통령이다.
E1) obama iss obama's president , who obama's president , has a history of t

In [34]:
# 최종 결과 확인
best_result = saved_results[7]

print(f"Selected Epoch: {best_result['epoch']} | Loss: {best_result['loss']:.4f}\n")

for i, item in enumerate(best_result["translations"], 1):
    print(f"K{i}) {item['source']}")
    print(f"E{i}) {item['prediction']} <end>")

Selected Epoch: 40 | Loss: 3.6791

K1) 오바마는 대통령이다.
E1) obama iss obama's president , the , obama said he is the best president to help obama . <end>
K2) 시민들은 도시 속에 산다.
E2) thessss the streets , the-yardss . <end>
K3) 커피는 필요 없다.
E3) no one has beeneded for coffee ,s , and thess . <end>
K4) 일곱 명의 사망자가 발생했다.
E4) the least seven people died when the tornado hit the killed in the tornado , which was reported in the . <end>


In [ ]:
# 불러올 모델의 Epoch
LOAD_EPOCH = 34
load_model_path = os.path.join(model_dir, f'seq2seq_model_epoch_{LOAD_EPOCH:02d}.pt')

# 기존과 같은 크기의 모델 생성
tested_model = build_model(
    attention_type="luong",
    input_dim=INPUT_DIM,
    output_dim=OUTPUT_DIM,
    emb_dim=EMB_DIM,
    hid_dim=HID_DIM,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
    device=device
)

# 가중치를 불러와 주입
tested_model.load_state_dict(torch.load(load_model_path, map_location=device))

# 번역 test 진행
for i, sentence in enumerate(test_sentences, 1):
    translation, _ = translate_sentence(
        sentence, 
        encoder_tokenizer, 
        decoder_tokenizer, 
        tested_model,
        max_len=40
    )
    print(f"K{i}) {sentence}")
    print(f"E{i}) {translation} <end>")

K1) 오바마는 대통령이다.
E1) obama iss president obama to the obama president , who is obama president , <end>
K2) 시민들은 도시 속에 산다.
E2) thess ares tos are toing to the streets , thess . <end>
K3) 커피는 필요 없다.
E3) no matter how to eat coffee , , , no ones , , no ones , said . <end>
K4) 일곱 명의 사망자가 발생했다.
E4) the least seven people died when the tornado killed in a tornado , which killed dozens of people died . <end>


# 회고
Bahdanau와 Luong을 모두 구현한 뒤 각각을 실행해보았을 떄 확실히 Luong쪽이 실행시간이 짧았다.   
- 초기 세팅으로 Epoch당 1:40, 2:30으로 1.5배정도 속도차이가 나는 것을 확인
따라서 이후 epoch를 늘려 튜닝할 때는 Luong을 계속 사용하였다.   
하이퍼 파라미터 별로 보자면
1. VOCAB_SIZE
- 10000과 12000을 테스트 해보았는데 emb와 hid의 크기때문인기 데이터의 양때문인지 12000때의 loss와 번역 상태가 더 안좋았다. 따라서 10000을 이후 사용하였다.
2. BATCH_SIZE
- 128과 256을 사용했는데 256을 사용해 한번에 읽는 양을 늘리고 epoch를 같이 늘렸다. 그와 함께 lr도 조정했는데 epoch를 늘린 것이 학습이 잘되는 것같아 256으로 사용했다.
3. EMB_DIM
- 128, 256, 512 모두 사용해보았는데 최종적으론 크기가 큰 512일때가 시간은 걸려도 성능은 잘나왔다.
4. HID_DIM
- EMB의 2배크기를 사용했는데 위와 동일
5. N_LAYERS
- 처음엔 1, epoch를 늘리고 2를 사용했는데 단어 선택이 좀더 좋아진 것같아서 2를 사용했다.
6. DROPOUT
- 과적합 방지용으로 사용하였고 0.2와 0.5를 실행해보고 최종적으로 중간값인 0.35를 채택했다.
7. Learning Rate
- 처음엔 1e-4 -> 6e-4를 사용했는데 epoch를 늘리고 더 줄여 8e-4 -> 4e-4로 조정하였다. 
8. Teacher Foring Scheduling
- 초기에는 최소값을 0.1, 0.2로 낮게 잡았는데 스스로 학습하고 내놓는 결과가 반복적이라던가 이상해서 최소값을 0.5로 잡았다. decay로 줄이는 양도 처음엔 빠르게 0.03으로 줄이다가 0.15로 완만하게 줄이게 하였다.

생각보다 번역성능이 좋지 않았다. 데이터의 개수가 문제인건지 데이터 처리가 문제인건지 학습이 문제인건지는 알 수 없으나 해당 attention의 구조를 이해하고 넘어갔다는 점에서 의의를 가지고자한다.

###

VOCAB_SIZE = 10000
BATCH_SIZE = 128
EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
DROPOUT = 0.3
LR = 6e-4
TF_MIN = 0.3
TF_START = 1.0
TF_DECAY = 0.03

Epoch 1: 100%|██████████| 436/436 [01:55<00:00,  3.79it/s, loss=5.6] 

Epoch 1/20 | Train Loss: 6.1507 | Val Loss: 8.2182

Epoch 2: 100%|██████████| 436/436 [01:58<00:00,  3.69it/s, loss=5.26]

Epoch 2/20 | Train Loss: 5.4290 | Val Loss: 8.2713

Epoch 3: 100%|██████████| 436/436 [01:57<00:00,  3.70it/s, loss=5.03]

Epoch 3/20 | Train Loss: 5.1363 | Val Loss: 7.7218

Epoch 4: 100%|██████████| 436/436 [02:00<00:00,  3.62it/s, loss=4.99]

Epoch 4/20 | Train Loss: 4.9483 | Val Loss: 7.4741

Epoch 5: 100%|██████████| 436/436 [02:00<00:00,  3.62it/s, loss=4.71]

Epoch 5/20 | Train Loss: 4.8380 | Val Loss: 7.1910
Epoch 5
obama's campaign , the's campaign , and the's aing to the .
thes of thes , which have beeneded by the city of the city of the city , where they were in the city .
the's not to be aed to the , but it's not a very important .
the number of people were killed in the city of the city's capital , which was killed in the city of the city's capital , according to the ministry of

Epoch 6: 100%|██████████| 436/436 [01:59<00:00,  3.64it/s, loss=4.72]

Epoch 6/20 | Train Loss: 4.7552 | Val Loss: 7.2426

Epoch 7: 100%|██████████| 436/436 [01:58<00:00,  3.67it/s, loss=4.85]

Epoch 7/20 | Train Loss: 4.7180 | Val Loss: 6.9729

Epoch 8: 100%|██████████| 436/436 [01:57<00:00,  3.71it/s, loss=4.47]

Epoch 8/20 | Train Loss: 4.6793 | Val Loss: 6.9635

Epoch 9: 100%|██████████| 436/436 [01:57<00:00,  3.72it/s, loss=4.78]

Epoch 9/20 | Train Loss: 4.6664 | Val Loss: 6.7758

Epoch 10: 100%|██████████| 436/436 [01:47<00:00,  4.05it/s, loss=4.46]

Epoch 10/20 | Train Loss: 4.6548 | Val Loss: 6.7616
Epoch 10
the obama is a to be the president's nomination .
the people are in the city of the city of java , where they are in the city of the city .
the's not a good thing , but it's not a good thing .
the death toll was killed in a series of deaths in the .

Epoch 11: 100%|██████████| 436/436 [01:48<00:00,  4.00it/s, loss=4.56]

Epoch 11/20 | Train Loss: 4.6555 | Val Loss: 6.6628

Epoch 12: 100%|██████████| 436/436 [01:52<00:00,  3.89it/s, loss=4.54]

Epoch 12/20 | Train Loss: 4.6535 | Val Loss: 6.6206

Epoch 13: 100%|██████████| 436/436 [01:51<00:00,  3.92it/s, loss=4.43]

Epoch 13/20 | Train Loss: 4.6674 | Val Loss: 6.5726

Epoch 14: 100%|██████████| 436/436 [01:55<00:00,  3.79it/s, loss=4.54]

Epoch 14/20 | Train Loss: 4.6629 | Val Loss: 6.5153

Epoch 15: 100%|██████████| 436/436 [01:59<00:00,  3.63it/s, loss=4.54]

Epoch 15/20 | Train Loss: 4.6962 | Val Loss: 6.4914
Epoch 15
obama's office is aing to obama's nomination .
thess are in the and ands in thes of the city of the city of the city of java .
the's not easy to be aed to , but it's not easy to be .
the death toll toll was killed in the death toll in the , , of the death toll .

Epoch 16: 100%|██████████| 436/436 [01:58<00:00,  3.67it/s, loss=4.44]

Epoch 16/20 | Train Loss: 4.7025 | Val Loss: 6.4556

Epoch 17: 100%|██████████| 436/436 [02:02<00:00,  3.56it/s, loss=4.77]

Epoch 17/20 | Train Loss: 4.7227 | Val Loss: 6.3541

Epoch 18: 100%|██████████| 436/436 [02:03<00:00,  3.53it/s, loss=4.61]

Epoch 18/20 | Train Loss: 4.7526 | Val Loss: 6.3484

Epoch 19: 100%|██████████| 436/436 [01:59<00:00,  3.65it/s, loss=4.79]

Epoch 19/20 | Train Loss: 4.7625 | Val Loss: 6.3537

Epoch 20: 100%|██████████| 436/436 [02:01<00:00,  3.58it/s, loss=4.44]

Epoch 20/20 | Train Loss: 4.7932 | Val Loss: 6.2874
Epoch 20
obama's a to the president , obama's president .
thesssssss in the areas of the mountainous locations .
the's not easy to be coffee , but the's not easy to be .
a death toll was killed in the death toll , the death toll , the .

VOCAB_SIZE = 10000
BATCH_SIZE = 256
EMB_DIM = 512
HID_DIM = 1024
N_LAYERS = 2
DROPOUT = 0.2
LR = 0.0004
TF_MIN = 0.5
TF_START = 1.0
TF_DECAY = 0.015

Epoch 1: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=5.49]
Epoch 1/40 | Train Loss: 6.0553 | Val Loss: 8.3734
Epoch 2: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=4.91]
Epoch 2/40 | Train Loss: 5.2051 | Val Loss: 8.2321
Epoch 3: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=4.83]
Epoch 3/40 | Train Loss: 4.8484 | Val Loss: 8.0930
Epoch 4: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=4.63]
Epoch 4/40 | Train Loss: 4.6099 | Val Loss: 7.8964
Epoch 5: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=4.36]
Epoch 5/40 | Train Loss: 4.4161 | Val Loss: 7.9277
Epoch 5
obama is aing to win the democratic nomination , but the president is aing to be a great .
the victims are in the city of the city's most famously populated , which is aed to the streets of the city .
the company is not known as a result of the 100 million , but not to be able to do .
the death toll of the death toll was killed in the deaths of the quake , which has been killed in the death toll .
Epoch 6: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=4.35]
Epoch 6/40 | Train Loss: 4.2736 | Val Loss: 7.8095
Epoch 7: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=4.19]
Epoch 7/40 | Train Loss: 4.1251 | Val Loss: 7.6894
Epoch 8: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=4.23]
Epoch 8/40 | Train Loss: 4.0437 | Val Loss: 7.4218
Epoch 9: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.86]
Epoch 9/40 | Train Loss: 3.9139 | Val Loss: 7.5361
Epoch 10: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.79]
Epoch 10/40 | Train Loss: 3.8361 | Val Loss: 7.4702
Epoch 10
obama is aing to the president , who is a president-elected president , who is taking office .
thes are scattered in the streets of the city , where they are taking place in the city's capital .
there are no signs of money , but not a 100 , 000 .
more than 1 , 000 people were killed and the death toll was killed , but the death toll was killed .
Epoch 11: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.65]
Epoch 11/40 | Train Loss: 3.7676 | Val Loss: 7.4289
Epoch 12: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.98]
Epoch 12/40 | Train Loss: 3.6738 | Val Loss: 7.4227
Epoch 13: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.51]
Epoch 13/40 | Train Loss: 3.6125 | Val Loss: 7.4843
Epoch 14: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.7] 
Epoch 14/40 | Train Loss: 3.5738 | Val Loss: 7.2847
Epoch 15: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.91]
Epoch 15/40 | Train Loss: 3.5250 | Val Loss: 7.4274
Epoch 15
obama iss president , who has become a president , who iss his life for president , who iss his life .
thes are  to take place in the streets of the city , where they are  to be ed .
instead , the can be no longer-free-s , but it can't be able to get a 
more than 1 , 000 people were killed , including a rise in the death toll , which has been killed .
Epoch 16: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=4.01]
Epoch 16/40 | Train Loss: 3.4564 | Val Loss: 7.3533
Epoch 17: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.65]
Epoch 17/40 | Train Loss: 3.4115 | Val Loss: 7.4346
Epoch 18: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.17]
Epoch 18/40 | Train Loss: 3.3812 | Val Loss: 7.3863
Epoch 19: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=2.82]
Epoch 19/40 | Train Loss: 3.3390 | Val Loss: 7.3459
Epoch 20: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.25]
Epoch 20/40 | Train Loss: 3.3164 | Val Loss: 7.3178
Epoch 20
obama hass his support for obama , who iss his in his country .
residents are  to take off the streets of the valley to the streets of the city , where they are in place .
no longer does not take a  of them , but thes no needs .
more than 40 people were killed and the death toll , which resulted in the death toll .
Epoch 21: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.05]
Epoch 21/40 | Train Loss: 3.2448 | Val Loss: 7.3514
Epoch 22: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.18]
Epoch 22/40 | Train Loss: 3.2482 | Val Loss: 7.3202
Epoch 23: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.4] 
Epoch 23/40 | Train Loss: 3.2573 | Val Loss: 7.3073
Epoch 24: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.11]
Epoch 24/40 | Train Loss: 3.2202 | Val Loss: 7.3714
Epoch 25: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=2.84]
Epoch 25/40 | Train Loss: 3.1685 | Val Loss: 7.3696
Epoch 25
obama has become a president , obama said his hisd his predecessors , who are obama , who is said .
residents are  to the streets of thes , which are forbidden streets in the streets .
instead , no needs to take a  of them , but it's no more fun .
more than 40 people are dead , the death toll , which issed with deaths , which killed 22 people .
Epoch 26: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.73]
Epoch 26/40 | Train Loss: 3.1726 | Val Loss: 7.2911
Epoch 27: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.5] 
Epoch 27/40 | Train Loss: 3.1482 | Val Loss: 7.3430
Epoch 28: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.41]
Epoch 28/40 | Train Loss: 3.1327 | Val Loss: 7.3209
Epoch 29: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.29]
Epoch 29/40 | Train Loss: 3.1312 | Val Loss: 7.2568
Epoch 30: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=2.98]
Epoch 30/40 | Train Loss: 3.1297 | Val Loss: 7.2666
Epoch 30
obama hass his hisd with , obama said . his inauguration is ready for obama .
residents are forbidden in the streets of the city ofs capital , where the s of places each .
instead , thes are not need to do a lot of money to do without a needs .
more than 40 people are killed , which are risen to a wave of deaths , which has risen to 55 , .
Epoch 31: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.23]
Epoch 31/40 | Train Loss: 3.1270 | Val Loss: 7.2051
Epoch 32: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=2.92]
Epoch 32/40 | Train Loss: 3.1087 | Val Loss: 7.2520
Epoch 33: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.11]
Epoch 33/40 | Train Loss: 3.1436 | Val Loss: 7.1474
Epoch 34: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=2.64]
Epoch 34/40 | Train Loss: 3.0987 | Val Loss: 7.2101
Epoch 35: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.1] 
Epoch 35/40 | Train Loss: 3.0657 | Val Loss: 7.2932
Epoch 35
obama hass his first his presidency , obama said his his his second term . obama is in his . .
residents in the  valley valley are often in the in the valley , which isss such as water .
instead , he's no need to do a  of money that can't take a money .
more than 40 people people have been killed and a result of the deaths , which is usually 55 people .
Epoch 36: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=2.85]
Epoch 36/40 | Train Loss: 3.0530 | Val Loss: 7.2627
Epoch 37: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.16]
Epoch 37/40 | Train Loss: 2.9969 | Val Loss: 7.2638
Epoch 38: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=2.86]
Epoch 38/40 | Train Loss: 2.9306 | Val Loss: 7.3385
Epoch 39: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=3.09]
Epoch 39/40 | Train Loss: 2.9245 | Val Loss: 7.3745
Epoch 40: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=3.37]
Epoch 40/40 | Train Loss: 2.8977 | Val Loss: 7.3421
Epoch 40
obama , who hass his his hisd his presidency , obama said his his becoming president .
residents in thes valleys , the each place in the valley , where a larger place in the valley .
don't need to take a lot of money , but it's not a lot of money to take a lot of money .
more than 40 people , the deaths of 55 people , and the death toll that rose to 55 , .

VOCAB_SIZE = 10000
BATCH_SIZE = 256
EMB_DIM = 512
HID_DIM = 1024
N_LAYERS = 2
DROPOUT = 0.5
LR = 0.0004
TF_MIN = 0.5
TF_START = 1.0
TF_DECAY = 0.015

Epoch 1: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=5.75]
Epoch 1/40 | Train Loss: 6.3165 | Val Loss: 7.5574
Epoch 2: 100%|██████████| 218/218 [02:22<00:00,  1.52it/s, loss=5.36]
Epoch 2/40 | Train Loss: 5.5879 | Val Loss: 8.4241
Epoch 3: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=5.12]
Epoch 3/40 | Train Loss: 5.3148 | Val Loss: 8.1647
Epoch 4: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=4.97]
Epoch 4/40 | Train Loss: 5.1066 | Val Loss: 7.8442
Epoch 5: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.87]
Epoch 5/40 | Train Loss: 4.9480 | Val Loss: 7.8452
Epoch 5
obama's democratic party has been a very important , but he is a good candidate .
the study is the most important-profile-seeeeee and theing the most important-profiles .
i'm not going to be a lot of the , but i'm not going to be a lot of the .
a suicide bomber killed a suicide bomber in the capital , killing across the blast , killing a bomb of the blast .
Epoch 6: 100%|██████████| 218/218 [02:23<00:00,  1.52it/s, loss=4.78]
Epoch 6/40 | Train Loss: 4.8281 | Val Loss: 7.5985
Epoch 7: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.63]
Epoch 7/40 | Train Loss: 4.7239 | Val Loss: 7.4870
Epoch 8: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.54]
Epoch 8/40 | Train Loss: 4.6543 | Val Loss: 7.4945
Epoch 9: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.82]
Epoch 9/40 | Train Loss: 4.5914 | Val Loss: 7.4007
Epoch 10: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.44]
Epoch 10/40 | Train Loss: 4.5309 | Val Loss: 7.3320
Epoch 10
obama is the president of the president , who is the president's president , said he was a president .
the workers are also in the area , and thes of thes are beinged to the streets , and thes of thes .
it is not clear how the problem is not , but it is not clear how the problem is not .
the death toll were killed in the deaths of the deaths , which killed at least 14 people .
Epoch 11: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.46]
Epoch 11/40 | Train Loss: 4.4994 | Val Loss: 7.2688
Epoch 12: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.34]
Epoch 12/40 | Train Loss: 4.4246 | Val Loss: 7.2104
Epoch 13: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.51]
Epoch 13/40 | Train Loss: 4.4172 | Val Loss: 7.1074
Epoch 14: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.59]
Epoch 14/40 | Train Loss: 4.3799 | Val Loss: 6.9613
Epoch 15: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.45]
Epoch 15/40 | Train Loss: 4.3465 | Val Loss: 7.0925
Epoch 15
i'm going to be the president to be the president's president , obama said .
thes are taking places in the streets , where they are very empty .
it's not a problem that doesn't have any specific damages , but it's not a problem .
at least seven people were killed in the death toll , the death toll in the death toll .
Epoch 16: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.43]
Epoch 16/40 | Train Loss: 4.3286 | Val Loss: 6.9663
Epoch 17: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.22]
Epoch 17/40 | Train Loss: 4.2859 | Val Loss: 6.9431
Epoch 18: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.26]
Epoch 18/40 | Train Loss: 4.2823 | Val Loss: 6.9370
Epoch 19: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.24]
Epoch 19/40 | Train Loss: 4.2982 | Val Loss: 6.7917
Epoch 20: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.01]
Epoch 20/40 | Train Loss: 4.2655 | Val Loss: 6.9064
Epoch 20
it's a to the president elected obama's nomination , obama said .
thess areeded to the streets , and thess of thess are empty .
it's not clear how many people can't have any otherwise , the otherwise , it's not necessary .
at least seven people were killed in the deaths , the deadly storm in the deaths , the deadly storm .
Epoch 21: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.6] 
Epoch 21/40 | Train Loss: 4.2589 | Val Loss: 6.7434
Epoch 22: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.18]
Epoch 22/40 | Train Loss: 4.2518 | Val Loss: 6.7674
Epoch 23: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.2] 
Epoch 23/40 | Train Loss: 4.2422 | Val Loss: 6.7593
Epoch 24: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.16]
Epoch 24/40 | Train Loss: 4.2444 | Val Loss: 6.7234
Epoch 25: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.44]
Epoch 25/40 | Train Loss: 4.2307 | Val Loss: 6.6737
Epoch 25
obama issing to the the , , obama said .
thess are theeded in the streets , and thes of thess in the streets of the city .
there are no immediates of the , , , which issed , the thes .
at least 67 people died in the deaths , the death toll in the death of the death .
Epoch 26: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.03]
Epoch 26/40 | Train Loss: 4.2096 | Val Loss: 6.7214
Epoch 27: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.17]
Epoch 27/40 | Train Loss: 4.2297 | Val Loss: 6.7220
Epoch 28: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.69]
Epoch 28/40 | Train Loss: 4.2621 | Val Loss: 6.6160
Epoch 29: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.55]
Epoch 29/40 | Train Loss: 4.2319 | Val Loss: 6.6936
Epoch 30: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.37]
Epoch 30/40 | Train Loss: 4.2357 | Val Loss: 6.6265
Epoch 30
obama wills obama wills to the the nomination , obama said .
the farmers aress are the to the thess , and thesssss .
the cannot be theed , thes , thess , , and thess , the cannot verify the .
at least 87 people died in the deaths sunday , the death toll in the deaths .
Epoch 31: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.12]
Epoch 31/40 | Train Loss: 4.2536 | Val Loss: 6.6397
Epoch 32: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.51]
Epoch 32/40 | Train Loss: 4.2658 | Val Loss: 6.5951
Epoch 33: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.24]
Epoch 33/40 | Train Loss: 4.2382 | Val Loss: 6.5806
Epoch 34: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.17]
Epoch 34/40 | Train Loss: 4.2472 | Val Loss: 6.5589
Epoch 35: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.44]
Epoch 35/40 | Train Loss: 4.2436 | Val Loss: 6.5638
Epoch 35
obama iss obama will bring a to the , obama said .
they are the to be in the the thes of thes , the the thess of thes .
it's not just that the cannot get the , , , , it's not just a tricky , , not just .
at least 19 people died in the deaths , the deaths were killed in the deaths .
Epoch 36: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.33]
Epoch 36/40 | Train Loss: 4.2174 | Val Loss: 6.5589
Epoch 37: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.2] 
Epoch 37/40 | Train Loss: 4.2079 | Val Loss: 6.5930
Epoch 38: 100%|██████████| 218/218 [02:22<00:00,  1.53it/s, loss=4.34]
Epoch 38/40 | Train Loss: 4.1805 | Val Loss: 6.5216
Epoch 39: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.01]
Epoch 39/40 | Train Loss: 4.0992 | Val Loss: 6.6393
Epoch 40: 100%|██████████| 218/218 [02:21<00:00,  1.54it/s, loss=4.32]
Epoch 40/40 | Train Loss: 4.1281 | Val Loss: 6.5601
Epoch 40
obama issing to obama's , , , , , , , .
thessss to the streets of the the thes , and thesss to the streets .
it's not just a , , , , , , , , , ,s , ,s , ,s , , which are not .
at least nine people died in the deaths of the deaths , sunday , the death toll .